<a href="https://colab.research.google.com/github/mxrch5th/2026-Visionaries-Kiosk/blob/main/recog_faces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =====================================================================
# [한글 파일 이름 자동 변환 기능 포함] 1번 셀: 데이터 수집 및 특징 추출
# =====================================================================

import sys
if 'deepface' not in sys.modules:
    print("DeepFace 설치 중...")
    !pip install deepface -q

from google.colab import files
from deepface import DeepFace
import numpy as np
import os

# 데이터 저장소 초기화 설정
if 'X_data' not in locals():
    X_data = []
if 'y_data' not in locals():
    y_data = []

print("==================================================")
print("▶ [한글 이름 허용 모드] 사진 등록을 시작합니다.")
print(f" 현재 누적된 데이터 개수: {len(X_data)}개")
print("==================================================")

while True:
    uploaded = files.upload()
    if not uploaded:
        print(f"▶ 사진 등록이 완료되었습니다. 총 {len(X_data)}개의 데이터가 누적되었습니다.")
        break

    for original_name in uploaded.keys():
        # 임시 영어 파일명 생성 (한글 이름을 안전한 영어 이름으로 변환)
        temp_english_name = "temp_kiosk_face_image.png"

        try:
            # 1. 사용자가 올린 한글 이름의 파일을 영어 이름으로 변경합니다.
            if os.path.exists(original_name):
                os.rename(original_name, temp_english_name)

            # 2. 영어로 바뀐 임시 파일을 인공지능에게 넘겨 특징을 뽑아냅니다.
            embedding_objs = DeepFace.represent(img_path=temp_english_name, model_name="VGG-Face", enforce_detection=False)
            face_feature = embedding_objs[0]["embedding"]

            # 3. 사용자에게는 원래 한글 이름을 보여주며 라벨링을 진행합니다.
            print(f"\n[사진 확인 완료]: {original_name}")
            print("0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)")
            answer = input("번호 입력 (0 또는 1): ")
            while answer not in ['0', '1']:
                answer = input("0 또는 1만 입력하세요: ")

            X_data.append(face_feature)
            y_data.append(int(answer))
            print(f"✔️ 누적 데이터: {len(X_data)}개")

        except Exception as e:
            print(f"⚠️ 에러 발생: {e}. 다른 사진을 사용해 주세요.")

        finally:
            # 뒷정리: 사용한 임시 파일과 원본 파일을 코랩에서 지워줍니다.
            if os.path.exists(temp_english_name):
                os.remove(temp_english_name)
            if os.path.exists(original_name):
                os.remove(original_name)

# =====================================================================
# [2번 셀] 머신러닝 모델 학습 실행
# =====================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

print("==================================================")
print("▶ [2번 셀] 커스텀 AI 모델 학습 (model.fit)")
print("==================================================")

# 1번 셀에서 데이터가 잘 넘어왔는지 확인합니다.
if 'X_data' not in locals() or len(X_data) < 2:
    print("❌ 오류: 1번 셀에서 사진 데이터를 먼저 2개 이상 등록하고 오셔야 합니다!")
else:
    # 데이터를 컴퓨터가 이해할 수 있는 배열로 변환합니다.
    X_train = np.array(X_data)
    y_train = np.array(y_data)

    # 인공지능 분류 모델(로지스틱 회귀)을 준비합니다.
    kiosk_ai_model = LogisticRegression(max_iter=1000)

    # ★★★ 이 명령어가 직접 모은 데이터로 AI를 '진짜 학습'시키는 코드입니다! ★★★
    kiosk_ai_model.fit(X_train, y_train)

    print("🎉 학습 완료! 50대 이상을 걸러내는 키오스크 전용 AI 모델이 빌드되었습니다.")

    # 우리가 만든 인공지능이 얼마나 똑똑한지 정확도를 측정해봅니다.
    y_pred = kiosk_ai_model.predict(X_train)
    accuracy = accuracy_score(y_train, y_pred) * 100
    print(f"📊 최종 학습 모델의 정확도: {accuracy:.1f}%")

▶ [한글 이름 허용 모드] 사진 등록을 시작합니다.
 현재 누적된 데이터 개수: 0개


Saving 1.png to 1.png


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5
To: /root/.deepface/weights/vgg_face_weights.h5


26-06-18 01:38:11 - 🔗 vgg_face_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5 to /root/.deepface/weights/vgg_face_weights.h5...


100%|██████████| 580M/580M [00:06<00:00, 87.5MB/s]


...

Saving 스크린샷 2026-06-18 111014.png to 스크린샷 2026-06-18 111014.png

[사진 확인 완료]: 스크린샷 2026-06-18 111014.png
0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)
번호 입력 (0 또는 1): 1
✔️ 누적 데이터: 67개


Saving 스크린샷 2026-06-18 111023.png to 스크린샷 2026-06-18 111023.png

[사진 확인 완료]: 스크린샷 2026-06-18 111023.png
0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)
번호 입력 (0 또는 1): 1
✔️ 누적 데이터: 68개


Saving 스크린샷 2026-06-18 111109.png to 스크린샷 2026-06-18 111109.png

[사진 확인 완료]: 스크린샷 2026-06-18 111109.png
0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)
번호 입력 (0 또는 1): 0
✔️ 누적 데이터: 69개


Saving 스크린샷 2026-06-18 111116.png to 스크린샷 2026-06-18 111116.png

[사진 확인 완료]: 스크린샷 2026-06-18 111116.png
0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)
번호 입력 (0 또는 1): 0
✔️ 누적 데이터: 70개


▶ 사진 등록이 완료되었습니다. 총 70개의 데이터가 누적되었습니다.
▶ [2번 셀] 커스텀 AI 모델 학습 (model.fit)
🎉 학습 완료! 50대 이상을 걸러내는 키오스크 전용 AI 모델이 빌드되었습니다.
📊 최종 학습 모델의 정확도: 95.7%


### 모델 저장하기

학습된 `kiosk_ai_model`을 `pickle` 라이브러리를 사용하여 파일로 저장할 수 있습니다. 이렇게 저장된 파일은 다른 환경에서도 모델을 다시 로드하여 사용할 수 있습니다.

In [6]:
import pickle

# 모델을 파일로 저장
model_filename = 'kiosk_ai_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(kiosk_ai_model, file)

print(f"모델이 '{model_filename}' 파일로 저장되었습니다.")

모델이 'kiosk_ai_model.pkl' 파일로 저장되었습니다.


### 저장된 모델 불러오기

저장된 모델 파일을 로드하여 다시 사용할 수 있습니다. `pickle.load()` 함수를 사용하면 됩니다.

In [7]:
import pickle

# 저장된 모델을 불러오기
model_filename = 'kiosk_ai_model.pkl'
with open(model_filename, 'rb') as file:
    loaded_model = pickle.load(file)

print(f"모델이 '{model_filename}' 파일에서 성공적으로 불러와졌습니다.")

# 불러온 모델을 사용하여 예측 예시 (X_data가 존재한다고 가정)
if 'X_data' in locals() and len(X_data) > 0:
    sample_prediction = loaded_model.predict(np.array([X_data[0]]))
    print(f"불러온 모델로 첫 번째 데이터 예측: {sample_prediction[0]}")
else:
    print("데이터가 없어서 예측 예시를 보여줄 수 없습니다.")

모델이 'kiosk_ai_model.pkl' 파일에서 성공적으로 불러와졌습니다.
불러온 모델로 첫 번째 데이터 예측: 1
